In [5]:
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"  
)
cur=conn.cursor()


In [10]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS patients (
        id                 SERIAL PRIMARY KEY,
        age                INTEGER CHECK (age BETWEEN 0 AND 120),
        gender             VARCHAR(10) NOT NULL,
        blood_type         VARCHAR(5),
        medical_condition  VARCHAR(100),
        admission_type     VARCHAR(50),
        medication         VARCHAR(100),
        billing_amount     NUMERIC(12,2) CHECK (billing_amount >= 0),
        test_results       VARCHAR(20) CHECK (test_results IN ('Normal','Abnormal','Inconclusive')),
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE (age, gender, blood_type, medical_condition, billing_amount)
    )
""")
conn.commit()
print("Table created ✓")

Table created ✓


In [15]:
conn.rollback()
print("Rollback done ✓")

Rollback done ✓


In [16]:
df = pd.read_csv("data\cleaned_data.csv")
df["Billing Amount"] = df["Billing Amount"].abs()

In [17]:
# list to tuples for batch insertion

records = [
    (
        row["Age"], row["Gender"], row["Blood Type"],
        row["Medical Condition"], row["Admission Type"],
        row["Medication"], round(row["Billing Amount"], 2),
        row["Test Results"]
    )
    for _, row in df.iterrows()
]

execute_values(cur, """
    INSERT INTO patients (age, gender, blood_type, medical_condition,
                          admission_type, medication, billing_amount, test_results)
    VALUES %s
    ON CONFLICT DO NOTHING
""", records, page_size=500)

conn.commit()
print(f"Inserted {cur.rowcount} rows ✓")

Inserted 466 rows ✓


In [18]:
# verify insertion
cur.execute("SELECT test_results, COUNT(*) FROM patients GROUP BY test_results")
for row in cur.fetchall():
    print(row)

('Abnormal', 18437)
('Inconclusive', 18198)
('Normal', 18331)


In [19]:
cur.close()
conn.close()
print("Connection closed ✓")

Connection closed ✓
